#1C. Recortar la imágenes con la máscara

Mejora el desempeño de los descriptores

In [ ]:
#Cargar librerias

import os
import shutil
import cv2
import numpy as np
from glob import glob
from tqdm import tqdm


In [ ]:
#Montar Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#Crear rutas
DATASET_PATH = "/content/drive/MyDrive/UNAL/Visión por computador/TareasDigiVision/Clasificación de Imágenes Médicas/results/chest_xray_preprocessed"
train_normal = glob(os.path.join(DATASET_PATH, "train/NORMAL/*.jpeg"))
train_pneu   = glob(os.path.join(DATASET_PATH, "train/PNEUMONIA/*.jpeg"))

print("NORMAL train:", len(train_normal))
print("PNEUMONIA train:", len(train_pneu))

MASKS_PATH  = "/content/drive/MyDrive/UNAL/Visión por computador/TareasDigiVision/Clasificación de Imágenes Médicas/results/masks"
OUTPUT_PATH = "/content/drive/MyDrive/UNAL/Visión por computador/TareasDigiVision/Clasificación de Imágenes Médicas/results/lungcropped"

NORMAL train: 1341
PNEUMONIA train: 3875


In [ ]:
# Reiniciar carpeta
if os.path.exists(OUTPUT_PATH):
    shutil.rmtree(OUTPUT_PATH)
os.makedirs(OUTPUT_PATH)

subsets = ["train", "val", "test"]
classes = ["NORMAL", "PNEUMONIA"]

for subset in subsets:
    for cls in classes:
        os.makedirs(os.path.join(OUTPUT_PATH, subset, cls), exist_ok=True)

print("Estructura creada en:", OUTPUT_PATH)

Estructura creada en: /content/drive/MyDrive/UNAL/Visión por computador/TareasDigiVision/Clasificación de Imágenes Médicas/results/lungcropped


In [ ]:
#Se define la función para aplicar la máscara

def apply_lung_mask(img_path, mask_path):
    """
    img_path: ruta de la imagen original
    mask_path: ruta de la máscara 512x512
    return: imagen recortada (solo pulmones)
    """
    # Imagen original en escala de grises
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

    # Redimensionar la imagen a 512x512 para que coincida con la máscara
    img_resized = cv2.resize(img, (512, 512), interpolation=cv2.INTER_AREA)

    # Leer máscara (0 o 255)
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

    # Aplicar máscara
    lung_only = cv2.bitwise_and(img_resized, img_resized, mask=mask)

    return lung_only

In [ ]:
#Se crea el nuevo dataset

for subset in subsets:
    for cls in classes:

        in_dir   = os.path.join(DATASET_PATH, subset, cls)
        mask_dir = os.path.join(MASKS_PATH, subset, cls)
        out_dir  = os.path.join(OUTPUT_PATH, subset, cls)

        img_paths  = glob(os.path.join(in_dir, "*.jpeg"))

        print(f"\nProcesando {subset}/{cls}: {len(img_paths)} imágenes")

        for p in tqdm(img_paths, desc=f"{subset}/{cls}"):
            fname = os.path.basename(p)
            mask_path = os.path.join(mask_dir, fname)

            if not os.path.exists(mask_path):
                print("Máscara NO encontrada:", mask_path)
                continue

            try:
                cropped = apply_lung_mask(p, mask_path)
            except Exception as e:
                print("ERROR en:", p, "->", e)
                continue

            out_path = os.path.join(out_dir, fname)
            cv2.imwrite(out_path, cropped)



Procesando train/NORMAL: 1341 imágenes


train/NORMAL: 100%|██████████| 1341/1341 [08:49<00:00,  2.53it/s]



Procesando train/PNEUMONIA: 3875 imágenes


train/PNEUMONIA: 100%|██████████| 3875/3875 [24:55<00:00,  2.59it/s]



Procesando val/NORMAL: 8 imágenes


...

val/PNEUMONIA: 100%|██████████| 8/8 [00:05<00:00,  1.55it/s]



Procesando test/NORMAL: 234 imágenes


test/NORMAL: 100%|██████████| 234/234 [01:27<00:00,  2.67it/s]



Procesando test/PNEUMONIA: 390 imágenes


test/PNEUMONIA: 100%|██████████| 390/390 [02:28<00:00,  2.63it/s]
